# Resize and Metadata Tag

### Tech:

Resize images to 512 x 512 to prevent excessive cost to GPT 4o mini

Run the model to capture several important metrics on the image such as damage_type, affected_system, location, urgency, estimated_repair_cost

Run the same model used for zoning laws - text-embedding-small-3. This will generate a vector embedding for the image that can be used for semantic search. I created a blog of all jsons 1 per image and stored as a new namespace in the same pinecone index

In [36]:
import os
import base64
import json
import time
import pickle

from PIL import Image
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_openai import OpenAIEmbeddings

# Load API key from .env file
load_dotenv()
client = OpenAI()  # Automatically uses OPENAI_API_KEY from environment
openai_api_key = os.getenv("OPENAI_API_KEY")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Connect to existing index
index = pc.Index("orlando-zoning-index")

In [3]:
def pad_and_resize(image_path: Path, output_path: Path, target_size: int = 512):
    """Pad image to square with white, then resize to target_size."""
    img = Image.open(image_path)
    
    # Convert to RGB if necessary (handles RGBA PNGs)
    if img.mode in ('RGBA', 'P'):
        img = img.convert('RGB')
    
    # Calculate padding to make square
    width, height = img.size
    max_dim = max(width, height)
    
    # Create white square canvas
    padded = Image.new('RGB', (max_dim, max_dim), (255, 255, 255))
    
    # Paste original image centered
    offset_x = (max_dim - width) // 2
    offset_y = (max_dim - height) // 2
    padded.paste(img, (offset_x, offset_y))
    
    # Resize to target
    resized = padded.resize((target_size, target_size), Image.LANCZOS)
    
    # Save as PNG
    resized.save(output_path, 'PNG')

In [7]:
# Setup paths
input_dir = Path('../data/prop-damage-images')
output_dir = Path('../data/prop-damage-images-resized-512')
output_dir.mkdir(parents=True, exist_ok=True)

In [8]:
# Process all PNG images
images = list(input_dir.glob('*.png'))
print(f"Found {len(images)} images to process")

Found 36 images to process


In [9]:
for img_path in images:
    output_path = output_dir / img_path.name
    pad_and_resize(img_path, output_path)
    print(f"Processed: {img_path.name}")

print(f"\nDone! Resized images saved to {output_dir}")

Processed: property_images_r1_c1_processed_by_imagy.png
Processed: property_images_r1_c2_processed_by_imagy.png
Processed: property_images_r1_c3_processed_by_imagy.png
Processed: property_images_r1_c4_processed_by_imagy.png
Processed: property_images_r1_c5_processed_by_imagy.png
Processed: property_images_r1_c6_processed_by_imagy.png
Processed: property_images_r2_c1_processed_by_imagy.png
Processed: property_images_r2_c2_processed_by_imagy.png
Processed: property_images_r2_c3_processed_by_imagy.png
Processed: property_images_r2_c4_processed_by_imagy.png
Processed: property_images_r2_c5_processed_by_imagy.png
Processed: property_images_r2_c6_processed_by_imagy.png
Processed: property_images_r3_c1_processed_by_imagy.png
Processed: property_images_r3_c2_processed_by_imagy.png
Processed: property_images_r3_c3_processed_by_imagy.png
Processed: property_images_r3_c4_processed_by_imagy.png
Processed: property_images_r3_c5_processed_by_imagy.png
Processed: property_images_r3_c6_processed_by_im

The plan is to capture the below

{
  "file_name": "damage_04.jpg",

  "damage_type": "Ceiling Water Stain",

  "affected_system": "plumbing",

  "location": "Master Bedroom",

  "severity": 7,

  "urgency": "immediate",

  "visible_area_affected": "medium",

  "estimated_repair_cost": "$800-$2000",

  "secondary_damage_risk": "mold growth, ceiling collapse",

  "maintenance_summary": "Active roof leak or plumbing failure likely. Visible mold growth."
  
}

### Damage Schema


In [ ]:
damage_schema = {
    "name": "property_damage_assessment",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "file_name": {
                "type": "string",
                "description": "Name of the image file"
            },
            "damage_type": {
                "type": "string",
                "description": "Type of damage observed (e.g., Water Stain, Roof Leak, HVAC Corrosion, etc)"
            },
            "affected_system": {
                "type": "string",
                "enum": ["roofing", "plumbing", "hvac", "electrical", "structural", "cosmetic"],
                "description": "Building system affected by the damage"
            },
            "location": {
                "type": "string",
                "description": "Likely location in property (e.g., Master Bedroom, Attic, Basement)"
            },
            "severity": {
                "type": "integer",
                "description": "Severity score from 1-10 (1=minor cosmetic, 10=critical structural)"
            },
            "urgency": {
                "type": "string",
                "enum": ["immediate", "short-term", "monitor"],
                "description": "How urgently repairs are needed"
            },
            "visible_area_affected": {
                "type": "string",
                "enum": ["small", "medium", "large"],
                "description": "Size of affected area: small (<1 sq ft), medium (1-10 sq ft), large (>10 sq ft)"
            },
            "estimated_repair_cost": {
                "type": "string",
                "description": "Rough cost range (e.g., $500-$1500)"
            },
            "secondary_damage_risk": {
                "type": "string",
                "description": "Potential consequences if untreated (e.g., mold growth, structural rot)"
            },
            "maintenance_summary": {
                "type": "string",
                "description": "Brief expert assessment and recommended action"
            }
        },
        "required": [
            "file_name",
            "damage_type",
            "affected_system",
            "location",
            "severity",
            "urgency",
            "visible_area_affected",
            "estimated_repair_cost",
            "secondary_damage_risk",
            "maintenance_summary"
        ],
        "additionalProperties": False
    }
}

In [11]:
def encode_image(image_path: Path) -> str:
    """Read image file and return base64 encoded string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [12]:
def analyze_damage_image(image_path: Path) -> dict:
    """
    Send image to GPT-4o mini and get structured damage assessment.
    Returns parsed JSON dict or raises exception on failure.
    """
    base64_image = encode_image(image_path)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are an expert property inspector assessing damage for real estate valuation.
                
                Analyze the image and provide a detailed damage assessment.
                
                Be specific about damage type, severity, and repair recommendations.
                
                Base severity scores on impact to property value:
                
                - 1-3: Minor cosmetic issues
                - 4-6: Moderate damage requiring attention
                - 7-9: Significant damage affecting value
                - 10: Critical structural/safety issue"""
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"Analyze this property damage image. The file name is: {image_path.name}"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}",
                            "detail": "high"  # Use high detail for damage assessment
                        }
                    }
                ]
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": damage_schema
        },
        max_tokens=500
    )
    
    # Parse and return the JSON response
    return json.loads(response.choices[0].message.content)

In [13]:
input_dir = Path("../data/prop-damage-images-resized-512")
output_dir = Path("../data/prop-damage-metadata")
output_dir.mkdir(parents=True, exist_ok=True)

In [17]:
# Collect any failures
failed_images = []

# Get all PNG images
images = list(input_dir.glob("*.png"))
print(f"Found {len(images)} images to process\n")

# Process each image with rate limiting
for i, img_path in enumerate(images, 1):
    print(f"[{i}/{len(images)}] Processing: {img_path.name}...", end=" ")
    
    try:
        # Analyze the image
        metadata = analyze_damage_image(img_path)
        
        # Save individual JSON file
        output_file = output_dir / f"{img_path.stem}_metadata.json"
        with open(output_file, "w") as f:
            json.dump(metadata, f, indent=2)
        
        print("✓ Done")
        
    except Exception as e:
        print(f"✗ Failed: {e}")
        failed_images.append({"file": img_path.name, "error": str(e)})
    
    # Rate limiting: wait 1 second between requests
    # Adjust if you hit rate limits (increase) or want faster processing (decrease)
    if i < len(images):
        time.sleep(1)

Found 36 images to process

[1/36] Processing: property_images_r1_c1_processed_by_imagy.png... ✓ Done
[2/36] Processing: property_images_r1_c2_processed_by_imagy.png... ✓ Done
[3/36] Processing: property_images_r1_c3_processed_by_imagy.png... ✓ Done
[4/36] Processing: property_images_r1_c4_processed_by_imagy.png... ✓ Done
[5/36] Processing: property_images_r1_c5_processed_by_imagy.png... ✓ Done
[6/36] Processing: property_images_r1_c6_processed_by_imagy.png... ✓ Done
[7/36] Processing: property_images_r2_c1_processed_by_imagy.png... ✓ Done
[8/36] Processing: property_images_r2_c2_processed_by_imagy.png... ✓ Done
[9/36] Processing: property_images_r2_c3_processed_by_imagy.png... ✓ Done
[10/36] Processing: property_images_r2_c4_processed_by_imagy.png... ✓ Done
[11/36] Processing: property_images_r2_c5_processed_by_imagy.png... ✓ Done
[12/36] Processing: property_images_r2_c6_processed_by_imagy.png... ✓ Done
[13/36] Processing: property_images_r3_c1_processed_by_imagy.png... ✓ Done
[14/36

In [18]:
# Summary
print(f"\n{'='*50}")
print(f"Completed! Metadata saved to: {output_dir}")
print(f"Successful: {len(images) - len(failed_images)}")
print(f"Failed: {len(failed_images)}")

# Show failures if any
if failed_images:
    print(f"\nFailed images:")
    for fail in failed_images:
        print(f"  - {fail['file']}: {fail['error']}")


Completed! Metadata saved to: ..\data\prop-damage-metadata
Successful: 36
Failed: 0


### Store in Pinecone

In [19]:
def metadata_to_text(metadata: dict) -> str:
    """
    Combine key fields into a single text for embedding.
    This creates a rich, searchable representation of the damage.
    """
    
    text = f"""
    Property Damage Assessment: {metadata['damage_type']}
    Affected System: {metadata['affected_system']}
    Location: {metadata['location']}
    Severity: {metadata['severity']}/10
    Urgency: {metadata['urgency']}
    Area Affected: {metadata['visible_area_affected']}
    Estimated Repair Cost: {metadata['estimated_repair_cost']}
    Secondary Damage Risk: {metadata['secondary_damage_risk']}
    Summary: {metadata['maintenance_summary']}
    """.strip()
    
    return text

In [ ]:
# Helper function to generate embedding
# Updated to use LangChain's OpenAIEmbeddings

def get_embedding(text: str) -> list:
    """Generate embedding using LangChain's OpenAIEmbeddings."""
    return embeddings.embed_query(text)

In [21]:
metadata_dir = Path("../data/prop-damage-metadata")
metadata_files = list(metadata_dir.glob("*_metadata.json"))

print(f"Found {len(metadata_files)} metadata files to process\n")

Found 36 metadata files to process



In [33]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Prepare vectors for upsert
vectors_to_upsert = []


for i, meta_file in enumerate(metadata_files, 1):
    print(f"[{i}/{len(metadata_files)}] Processing: {meta_file.name}...", end=" ")
    
    # Load metadata JSON
    with open(meta_file, "r") as f:
        metadata = json.load(f)
    
    # Create text blob for embedding
    text_blob = metadata_to_text(metadata)
    
    # Generate embedding
    embedding = get_embedding(text_blob)
    
    # Prepare vector with metadata for Pinecone
    # Using file_name (without extension) as the unique ID
    vector_id = meta_file.stem.replace("_metadata", "")
    
    vectors_to_upsert.append({
        "id": vector_id,
        "values": embedding,
        "metadata": metadata  # Store full metadata for retrieval
    })
    
    print("✓ Done")

print(f"\nPrepared {len(vectors_to_upsert)} vectors for upsert")

[1/36] Processing: property_images_r1_c1_processed_by_imagy_metadata.json... ✓ Done
[2/36] Processing: property_images_r1_c2_processed_by_imagy_metadata.json... ✓ Done
[3/36] Processing: property_images_r1_c3_processed_by_imagy_metadata.json... ✓ Done
[4/36] Processing: property_images_r1_c4_processed_by_imagy_metadata.json... ✓ Done
[5/36] Processing: property_images_r1_c5_processed_by_imagy_metadata.json... ✓ Done
[6/36] Processing: property_images_r1_c6_processed_by_imagy_metadata.json... ✓ Done
[7/36] Processing: property_images_r2_c1_processed_by_imagy_metadata.json... ✓ Done
[8/36] Processing: property_images_r2_c2_processed_by_imagy_metadata.json... ✓ Done
[9/36] Processing: property_images_r2_c3_processed_by_imagy_metadata.json... ✓ Done
[10/36] Processing: property_images_r2_c4_processed_by_imagy_metadata.json... ✓ Done
[11/36] Processing: property_images_r2_c5_processed_by_imagy_metadata.json... ✓ Done
[12/36] Processing: property_images_r2_c6_processed_by_imagy_metadata.json

In [34]:
batch_size = 100
namespace = "property-damage"

for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch, namespace=namespace)
    print(f"Upserted batch {i // batch_size + 1}: {len(batch)} vectors")

print(f"\n{'='*50}")
print(f"Done! Upserted {len(vectors_to_upsert)} vectors to namespace: '{namespace}'")

Upserted batch 1: 36 vectors

Done! Upserted 36 vectors to namespace: 'property-damage'


In [35]:
# Cell 6: Verify upload (optional) 

# Check index stats to confirm
stats = index.describe_index_stats()
print(f"\nIndex stats:")
print(f"  Total vectors: {stats['total_vector_count']}")
print(f"  Namespaces: {list(stats['namespaces'].keys())}")
for ns, data in stats['namespaces'].items():
    print(f"    - {ns}: {data['vector_count']} vectors")


Index stats:
  Total vectors: 2610
  Namespaces: ['property-damage', '__default__']
    - property-damage: 36 vectors
    - __default__: 2574 vectors


In [ ]:
# Save the vectors (with metadata) to a local .pkl file for backup

output_path = Path("../data/embeddings_prop_damage_resized_512.pkl")

# vectors_to_upsert already contains id, values, and metadata
with open(output_path, "wb") as f:
    pickle.dump(vectors_to_upsert, f)

print(f"Saved {len(vectors_to_upsert)} embeddings to: {output_path}")

Saved 36 embeddings to: ..\data\embeddings_prop_damage_resized_512.pkl
